## Steps of Feature Engineering
- Feature engineering starter 
- Target and Features Split
- Train Test Split
- Missing values handling
- Outliers detection and handing
- Categorical feature encoding
- Scaling
- Feature creation

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('dataset/titanic_data_updated.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### Remove unnecessary columns

In [ ]:
df = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
# df = df.drop(['Cabin'], axis=1)
df.sample(5)

### Separation of Target and Features

In [ ]:
X = df.drop('Survived', axis=1)
display(X.head())
y = df['Survived']
display(y.head())

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('X_train:', X_train.shape, '\nX_test:', X_test.shape)
print('y_train:', y_train.shape, '\ny_test:', y_test.shape)

## Imputing missing values

### Numerical missing value imputation using pandas

In [ ]:
X_train.head()

In [ ]:
df.isnull().sum()

In [ ]:
X_train.isnull().sum()

In [ ]:
X_test.isnull().sum()

In [ ]:
age_mean = X_train['Age'].mean()
age_median = X_train['Age'].median()
print('age_mean', age_mean)
print('age_median', age_median)

X_train['age_mean_imputer'] = X_train['Age'].fillna(age_mean) # imput only for X_train data
X_train['age_median_imputer'] = X_train['Age'].fillna(age_median)

X_train.info()

In [ ]:
import seaborn as sns

sns.kdeplot(X_train[['Age', 'age_mean_imputer', 'age_median_imputer']], fill=True)
# sns.kdeplot(X_train['age_mean_imputer'])

### Missing value imputation using sklearn

In [ ]:
from sklearn.impute import SimpleImputer

simpleImputer = SimpleImputer(missing_values=np.nan, strategy='mean')
simpleImputer

In [ ]:
simpleImputer.fit(X_train[['Age']])

In [ ]:
X_train['Age'] = simpleImputer.transform(X_train[['Age']])

In [ ]:
X_train.isnull().sum()

In [ ]:
X_train = X_train.drop(['age_mean_imputer', 'age_median_imputer'], axis=1)
X_train.columns

In [ ]:
age_median

In [ ]:
X_test.isnull().sum()

In [ ]:
X_test['Age'] = simpleImputer.transform(X_test[['Age']])
X_test.isnull().sum()

## Imputing missing values for categorical feature

In [ ]:
X_train['Embarked'].unique()

In [ ]:
print(X_train['Embarked'].isnull().sum())
print(X_test['Embarked'].isnull().sum())

In [ ]:
sns.countplot(X_train, x='Embarked')

In [ ]:
sns.countplot(X_test, x='Embarked')

In [ ]:
missing_embarked_indices = X_train.loc[X_train['Embarked'].isna()].index
X_train.loc[missing_embarked_indices]

In [ ]:
embarked_imputer = SimpleImputer(strategy='most_frequent')
embarked_imputer.fit(X_train[['Embarked']])

In [ ]:
X_train[['Embarked']] = embarked_imputer.transform(X_train[['Embarked']]) # ? should i use ravel() or not 
X_test[['Embarked']] = embarked_imputer.transform(X_test[['Embarked']])

X_train.loc[missing_embarked_indices]

## Categorical value imputation with 'missing' string and Indicator

In [ ]:
X_train['Cabin'].isnull().sum(), X_test['Cabin'].isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
sns.countplot(X_train, x='Cabin')
plt.xticks(rotation=90)
plt.show()

In [ ]:
cabin_imputer = SimpleImputer(
    missing_values = np.nan,
    strategy = 'constant',
    fill_value = 'missing',
    add_indicator = True
)

cabin_imputer.fit(X_train[['Cabin']])

In [ ]:
X_train[['Cabin', 'cabin_missing_indicator']] = cabin_imputer.transform(X_train[['Cabin']])

In [ ]:
X_test[['Cabin', 'cabin_missing_indicator']] = cabin_imputer.transform(X_test[['Cabin']])

In [ ]:
X_train.head()

In [ ]:
X_train.isnull().sum()

## Outlier Detection (using Z score)

In [ ]:
X_train.head()

In [ ]:
sns.kdeplot(X_train, x='Age', fill=True)

In [ ]:
age_mean = X_train['Age'].mean()
age_std = X_train['Age'].std()
age_mean, age_std

In [ ]:
X_train['age_z_score'] = (X_train['Age'] - age_mean) / age_std
X_train.head()

In [ ]:
sns.boxplot(X_train['age_z_score'])
X_train['age_z_score'].describe()

In [ ]:
age_outliers_zs = X_train[abs(X_train['age_z_score']) > 3]
age_outliers_zs

In [ ]:
sns.kdeplot(X_train['Fare'], fill=True)

In [ ]:
print(X_train['Fare'].describe())
sns.boxplot(X_train['Fare'])

In [ ]:
fare_mean = X_train['Fare'].mean()
fare_std = X_train['Fare'].std()

X_train['fare_zscore'] = (X_train['Fare'] - fare_mean) / fare_std
X_train.sample(5)

In [ ]:
X_train['fare_zscore'].describe()

In [ ]:
fare_outliers_zs = X_train[abs(X_train['fare_zscore']) > 3]
fare_outliers_zs.head()

## Outlier Detection using IQR

In [ ]:
sns.kdeplot(X_train['Age'])

In [ ]:
X_train['Age'].describe()

In [ ]:
age_Q1 = X_train['Age'].quantile(0.25)
age_Q3 = X_train['Age'].quantile(0.75)
print('quantiles:', age_Q1, age_Q3)

age_IQR = age_Q3 - age_Q1
print('IQR:', age_IQR)

age_min = age_Q1 - 1.5*age_IQR
age_max = age_Q3 + 1.5*age_IQR
print('min-max:', age_min, age_max)

In [ ]:
sns.boxplot(X_train['Age'])

In [ ]:
age_outliers_iqr = X_train[(X_train['Age'] < age_min) | (X_train['Age'] > age_max)]
print(age_outliers_iqr.shape)
age_outliers_iqr.sample(5)

In [ ]:
sns.kdeplot(X_train['Fare'])

In [ ]:
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)
print('q1, q3:', fare_Q1, fare_Q3)

fare_IQR = fare_Q3 - fare_Q1
print('fare IQR:', fare_IQR)

fare_min = fare_Q1 - fare_IQR * 1.5
fare_max = fare_Q3 + fare_IQR * 1.5
print('min-max:', fare_min, fare_max)

In [ ]:
sns.boxplot(X_train['Fare'])

In [ ]:
fare_outliers_iqr = X_train[(X_train['Fare'] < fare_min) | (X_train['Fare'] > fare_max)]
print(fare_outliers_iqr.shape)
fare_outliers_iqr